In [0]:
# Databricks notebook source
from pyspark.sql.functions import current_timestamp
import uuid

caminho_csv = "/Volumes/credit_risk/bronze/landing_zone/german_credit_data.csv"
tabela_bronze = "credit_risk.bronze.bronze_credit_risk"
execution_id = str(uuid.uuid4()) 

try:
    df_raw = spark.read.csv(caminho_csv, header=True, inferSchema=True)
    # Sanitize column names by replacing spaces with underscores
    for col_name in df_raw.columns:
        df_raw = df_raw.withColumnRenamed(col_name, col_name.replace(" ", "_"))
    df_bronze = df_raw.withColumn("data_ingestao", current_timestamp())

    df_bronze.write.format("delta").mode("append").saveAsTable(tabela_bronze)
    
    spark.sql(f"""
        INSERT INTO credit_risk.bronze.log_pipeline_execution 
        VALUES ('{execution_id}', '01_bronze_ingestion', '{tabela_bronze}', {df_bronze.count()}, 'SUCCESS', '', current_timestamp())
    """)
    print("Ingestão Bronze concluída com sucesso.")

except Exception as e:
    erro = str(e).replace("'", "")
    spark.sql(f"""
        INSERT INTO credit_risk.bronze.log_pipeline_execution 
        VALUES ('{execution_id}', '01_bronze_ingestion', '{tabela_bronze}', 0, 'FAILED', '{erro}', current_timestamp())
    """)
    raise e